# QTrack 2026 — Level 1: Entanglement & correlations

**Tri-Co Hackathon · Bi-Co Quantum Computing Club**

**Mission:** Create Bell and GHZ states, compare measurements in different bases, and distinguish a relative phase.

**Prerequisites:** Level 0 ideas; basic Python.  
**Deliverable:** A completed `.ipynb` notebook with your code, results, and explanations.

Follow the numbered tasks, implement the `TODO` functions, and use the self-checks to test your work.


## Getting started

Run the next cell. In Google Colab, upload this notebook and choose **Runtime → Run all** after completing your tasks. The first run may install Python packages using the internet. After installation, all circuit simulations run locally on the CPU.

**Bit order convention:** Qiskit writes the most significant bit on the **left** of a bitstring. For example `"10"` means qubit 1 is `1` and qubit 0 is `0`. Our checks follow this convention.

**If you get `ModuleNotFoundError`:** rerun the installation cell or restart the notebook kernel and rerun.

In [74]:
# First run: installs dependencies only if missing (requires internet).
import sys, subprocess, importlib.util
if importlib.util.find_spec("qiskit") is None or importlib.util.find_spec("qiskit_aer") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qiskit>=2.5,<3", "qiskit-aer>=0.17,<0.19", "numpy>=1.26,<3", "matplotlib>=3.8"])
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService
print("Qiskit ready. Free local simulation only; no IBM account or paid hardware required.")

Qiskit ready. Free local simulation only; no IBM account or paid hardware required.


### How to work
1. Read the task and its expected result.
2. Replace `return None` with your implementation.
3. Run the **self-check** underneath. A passing check verifies an observable property but is not a complete grading decision.
4. Answer the **Explain** questions in Markdown, using your own words.
5. Run the notebook top to bottom before exporting. Do not delete your code, plots or outputs.

## Worked example — two-qubit product state

Two Hadamard gates independently create equal probabilities for `00`, `01`, `10` and `11`. That is **not** a Bell pair, because the qubits are not correlated.

In [75]:
product = QuantumCircuit(2)
product.h([0,1])
print(product.draw(output="text"))
print(Statevector.from_instruction(product).probabilities_dict())

     ┌───┐
q_0: ┤ H ├
     ├───┤
q_1: ┤ H ├
     └───┘
{np.str_('00'): np.float64(0.2499999999999999), np.str_('01'): np.float64(0.2499999999999999), np.str_('10'): np.float64(0.2499999999999999), np.str_('11'): np.float64(0.2499999999999999)}


### Task 1A — Bell pair

Create an entangled Bell pair. **Expected:** `00` and `11` each have probability 0.5; other outcomes 0.

In [9]:
def bell_plus() -> QuantumCircuit:
    """Prepare (|00⟩+|11⟩)/sqrt(2)."""
    qc = QuantumCircuit(2) # create 2 qubits that both start at 0
    qc.h(0) # apply Hadamard gate to qubit 0
    qc.cx(0, 1) # apply CNOT gate with qubit 0 as control and qubit 1 as target
    return qc # return the circuit

In [10]:
# Visible self-check (not a hidden grader)
if bell_plus() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    p = Statevector.from_instruction(bell_plus()).probabilities_dict()
    assert abs(p.get('00',0)-.5)<1e-10 and abs(p.get('11',0)-.5)<1e-10 and p.get('01',0)<1e-10
    print("Self-check passed.")

Self-check passed.


### Task 1B — GHZ state

Generalize to three and four entangled qubits. **Expected:** only all-zero and all-one bitstrings remain.

In [78]:
def ghz_state(n: int = 3) -> QuantumCircuit:
    """Prepare an n-qubit GHZ state (|0...0⟩+|1...1⟩)/sqrt(2); support n=3,4."""
    circuit = QuantumCircuit(n)
    circuit.h(0)
    for i in range(n-1):
        circuit.cx(i,i+1)
    print(circuit)
    return circuit

In [79]:
# Visible self-check (not a hidden grader)
if ghz_state() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    p = Statevector.from_instruction(ghz_state(3)).probabilities_dict()
    assert abs(p.get('000',0)-.5)<1e-10 and abs(p.get('111',0)-.5)<1e-10
    print("Self-check passed.")

     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘
     ┌───┐          
q_0: ┤ H ├──■───────
     └───┘┌─┴─┐     
q_1: ─────┤ X ├──■──
          └───┘┌─┴─┐
q_2: ──────────┤ X ├
               └───┘
Self-check passed.


### Task 1C — Relative phase

Create Bell-minus. **Expected:** measuring directly in the computational (Z) basis cannot distinguish Bell-plus from Bell-minus.

In [80]:
def bell_minus() -> QuantumCircuit:
    """Prepare (|00⟩-|11⟩)/sqrt(2)."""
    circuit = QuantumCircuit(2)
    circuit.h(0)
    circuit.z(0)
    circuit.cx(0,1)
    return circuit

In [81]:
# Visible self-check (not a hidden grader)
if bell_minus() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    p = Statevector.from_instruction(bell_minus()).probabilities_dict()
    assert abs(p.get('00',0)-.5)<1e-10 and abs(p.get('11',0)-.5)<1e-10
    print("Self-check passed.")

Self-check passed.


### Task 1D — Test correlations in two bases

Rotate both qubits with H before measuring in X basis. **Expected:** Bell-plus is correlated in X; Bell-minus is anticorrelated in X.

In [82]:
def bell_counts(kind: str = "plus", basis: str = "Z", shots: int = 1024, seed: int = 11) -> dict:
    """Return measurement counts for Bell-plus or Bell-minus in Z or X basis."""
    circuit = QuantumCircuit(2)
    circuit.h(0)
    if kind == "minus":
        circuit.z(0)
    circuit.cx(0,1)
    if basis == "X":
        circuit.h(0)
        circuit.h(1)
    circuit.measure_all()

    backend = AerSimulator(seed_simulator=seed)

    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(circuit)

    sampler = Sampler(mode=backend)

    job = sampler.run([isa_circuit], shots=shots)
    result = job.result()
    
    return result[0].data.meas.get_counts()

In [83]:
# Visible self-check (not a hidden grader)
if bell_counts() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    counts = bell_counts('minus','X',1024,11)
    assert sum(counts.values())==1024
    assert (counts.get('01',0)+counts.get('10',0))/1024>.9
    print("Self-check passed.")

Self-check passed.


## Task 1E — Interpretation (required, 100–200 words)

1. Why are Z-basis counts alone unable to distinguish Bell-plus and Bell-minus?
2. Explain what changes after measuring both qubits in the X basis.
3. Why does a uniform 4-outcome distribution from independent Hadamard gates not demonstrate entanglement?
4. Report your `Z` and `X` counts for both Bell states below.

**Write your explanation here.**

## Completion checklist
- [ ] Bell-plus, GHZ and Bell-minus built
- [ ] Z- and X-basis data shown for both Bell states
- [ ] Correlation interpretation written
- [ ] Self-checks pass and notebook runs top to bottom

**Optional stretch:** Calculate the parity expectation `(P00+P11) − (P01+P10)` in each basis and interpret its sign.